# 05 — Evaluation (CRISP-DM)

Avaliação honesta do campeão publicado pelo notebook 04. Três compromissos:

1. **Factual vs. contrafactual** — separar o que é medido (demanda observada vs.
   prevista) do que é estimado (lucro sob descontos que não ocorreram).
2. **Auditoria de vazamento re-verificada** — o guardrail é checado *aqui também*,
   não só no treino: nenhuma coluna vetada pode ser feature.
3. **Contexto de negócio** — comparar com baselines ingênuos e com a curva de
   desconto, para calibrar a expectativa do ganho.

Consumimos apenas artefatos publicados (`models/`), sem re-treinar nada.

## 1) Artefatos e proveniência

O snapshot de métricas e o cartão de modelo carregam a proveniência do run
(seed, hash dos dados, janelas temporais), garantindo reprodutibilidade.

In [ ]:
import json

import pandas as pd

from src.config.paths import MODELS_DIR

snapshot = json.loads((MODELS_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
cartao = json.loads((MODELS_DIR / "model_card.json").read_text(encoding="utf-8"))

prov = snapshot["proveniencia"]
print(f"Modelo campeão : {snapshot['modelo']}")
print(f"Seed           : {prov['seed']} | hash dos dados: {prov['hash_dados']}")
print("Linhas por split:", prov["linhas_por_split"])
for nome, (inicio, fim) in prov["periodos"].items():
    print(f"  {nome:>9}: {inicio} → {fim}")

## 2) Auditoria de vazamento re-verificada

O contrato de features é validado *fail-closed* também na avaliação. Se qualquer
coluna vetada por vazamento (`quantidade_vendida_mes`, `estoque_disponivel`,
`custo_marketing_unitario`, `ruptura_flag`) ou o próprio alvo aparecer como
feature, a célula aborta.

In [ ]:
from src.modeling import train_pipeline as tp

base = tp.carregar_base_preparada()
tp.validar_contrato_features(base)  # fail-closed: vazamento aborta o notebook

vetadas = set(tp.colunas_com_vazamento())
features = set(cartao["features_numericas"]) | set(cartao["features_categoricas"])
assert not (features & vetadas), f"Vazamento no contrato: {sorted(features & vetadas)}"
assert cartao["alvo"] not in features, "O alvo não pode ser feature."
assert features <= set(base.columns), "Features do cartão ausentes na base."

print(f"Guardrail OK — {len(features)} features leakage-safe contra {len(vetadas)} colunas vetadas.")
print("Colunas vetadas:", sorted(vetadas))

## 3) Métricas factuais vs. baselines ingênuos

O RMSE do modelo é comparado com duas referências sem modelo: prever a média
global do treino e prever a média por (loja, produto). É o teste de honestidade
mínimo — mostrar que o modelo agrega valor além do trivial.

In [ ]:
teste = snapshot["teste"]
baselines = snapshot["baselines"]

comparativo = pd.DataFrame(
    [
        {"referência": f"modelo ({snapshot['modelo']})", "rmse": teste["rmse"]},
        {"referência": "baseline média global", "rmse": baselines["rmse_media_global"]},
        {"referência": "baseline média por SKU", "rmse": baselines["rmse_media_sku"]},
    ]
)
comparativo["ganho_vs_modelo_%"] = (comparativo["rmse"] / teste["rmse"] - 1.0) * 100.0
comparativo.round(4)

In [ ]:
print(f"R² (teste) : {teste['r2']:.3f}")
print(f"MAE (teste): {teste['mae']:.3f}  (≈ {teste['mae_ratio']:.1%} da demanda média)")
print(f"RMSE (teste): {teste['rmse']:.3f} vs. média por SKU {baselines['rmse_media_sku']:.3f}")

## 4) Honestidade do contrafactual

O ganho de lucro da política é uma **estimativa** (a demanda sob descontos que
não ocorreram é predita pelo modelo). Reportamos validação e teste lado a lado,
com a nota de honestidade gravada no snapshot.

In [ ]:
colunas = ["lucro_baseline", "lucro_politica", "uplift_pct", "desconto_medio", "pct_linhas_com_desconto"]
negocio = pd.DataFrame(
    {
        "validacao": {k: snapshot["validacao"][k] for k in colunas},
        "teste": {k: teste[k] for k in colunas},
    }
)
print(snapshot["honestidade"])
negocio.round(4)

## 5) Curva de negócio (demanda/receita/lucro por desconto)

A curva **média agregada** é monotonicamente decrescente: no agregado, o preço
cheio já é o melhor. O uplift da política vem da **heterogeneidade por linha** —
em SKUs/dias específicos, um desconto pequeno compra volume suficiente para
superar a queda de margem. É exatamente essa diferença que o otimizador captura.

In [ ]:
curva = pd.read_csv(MODELS_DIR / "business_curve.csv")
curva.round(4)

In [ ]:
monotonica = curva["lucro_medio_previsto"].is_monotonic_decreasing
otimo_medio = curva.loc[curva["lucro_medio_previsto"].idxmax()]
print(f"Curva média monotônica decrescente: {monotonica}")
print(f"Ótimo médio agregado: desconto={otimo_medio['desconto_pct']:.2f}")
print(f"Demanda média: {curva['qtd_media_prevista'].iloc[0]:.2f} (0%) → {curva['qtd_media_prevista'].iloc[-1]:.2f} (12%)")

## 6) Importância das features

Importâncias legíveis por feature original (bloco one-hot somado de volta),
para leitura de negócio. `promocao_tipo` domina porque é a alavanca de volume
mais forte da base; as variáveis de preço vêm em seguida, coerentes com o
framing de elasticidade.

In [ ]:
importancias = pd.Series(cartao["importancia_features"], name="importancia").sort_values(ascending=False)
importancias.head(12).round(4)

## 7) Casos por linha no teste

O detalhamento por linha do Q4/2025 mostra *onde* a política age: quantas linhas
recebem desconto e o lucro médio com e sem a política.

In [ ]:
cenarios = pd.read_parquet(MODELS_DIR / "test_predictions.parquet")
print(f"Casos no teste: {len(cenarios):,}")
cenarios.head(8).round(3)

In [ ]:
cenarios["recebeu_desconto"] = cenarios["desconto_otimo"] > 0
resumo = cenarios.groupby("recebeu_desconto")[["lucro_politica", "lucro_baseline"]].mean().round(3)
print(f"Linhas com desconto: {cenarios['recebeu_desconto'].mean():.1%}")
resumo

## Conclusão

- **Factual**: o modelo reduz o RMSE de demanda bem abaixo dos baselines
  ingênuos (média global e média por SKU) no teste Q4 — valor preditivo real.
- **Contrafactual**: o uplift de lucro da política é uma **estimativa**,
  reportada com nota de honestidade e calibrada contra o preço de tabela.
- **Vazamento**: guardrail de features re-verificado na avaliação; nenhuma
  coluna vetada entra no modelo.

A Fase 4 consome estes mesmos artefatos (`business_curve.csv`,
`metrics_snapshot.json`, `test_predictions.parquet`, `best_model.pkl`) para o app
e o site, sem re-treinar.